# NBA 3.1 — HCP Universe Identification v3 (Product-Class TRx Decile Approach)

Builds the HCP Universe by computing **per-product-class TRx cumulative deciles** (matching v1/NBA 3.0 methodology) and combining the Decile 2+ sets.

| # | Step | Logic |
|---|------|-------|
| 1 | Per-class TRx decile | For each of SGLT2, GLP1, ACE, ARB, sMRA: count TRx (claims) from `pt_lvl_trx`, cumulative TRx decile (~10% volume per decile), keep Decile 2+ |
| 2 | Union / Intersection | Union (any of 5) or Intersection (all 5) of Decile 2+ sets |
| 3 | Kerendia Writers | Union in all Kerendia writers (full history) |
| 4 | Specialty HCPs | NEPH, CARD, ENDO, PCP, HOSPITALIST from `lrx_nbrx` |
| 5 | Targeting & BPT | From `CP_KER_HCP` (unchanged) |

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql import DataFrame
from functools import reduce
from datetime import datetime as dt

# ── Catalog / Schema ─────────────────────────────────────────────────────────
CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
SCHEMA  = "pace_ker_in_house_schema"

# ── Lookback window ─────────────────────────────────────────────────────
LOOKBACK_START = "2024-10-01"
LOOKBACK_END   = "2026-09-30"

# ── Product classes for per-class deciling ───────────────────────────────────
TARGET_PRODUCT_CLASSES = ["SGLT2", "GLP1", "ACE", "ARB", "sMRA"]
DECILE_CUTOFF = 2

# ── Specialties for Criterion 4 ──────────────────────────────────────────
TARGET_SPECIALTIES = ["NEPH", "CARD", "ENDO", "PCP", "HOSPITALIST"]

# ── Output path ───────────────────────────────────────────────────────────────
OUTPUT_PATH = (
    f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_volume/"
    "hcp_level_model/dev_folder_3_1/notebook_output/"
    "kerendia_nba_3_hcp_universe_v3"
)

print(f"Lookback        : {LOOKBACK_START} → {LOOKBACK_END}")
print(f"Product classes : {TARGET_PRODUCT_CLASSES}")
print(f"Decile cutoff   : {DECILE_CUTOFF}+")
print(f"Output path     : {OUTPUT_PATH}")

In [0]:
# ── lrx_nbrx (for Kerendia writers & specialty HCPs) ───────────────────────────────
lrx_nbrx = spark.table(f"`{CATALOG}`.`{SCHEMA}`.lrx_nbrx")

lrx_window = lrx_nbrx.filter(
    (F.col("WEEK_ENDING_DATE") >= LOOKBACK_START) &
    (F.col("WEEK_ENDING_DATE") <= LOOKBACK_END)
)

# ── pt_lvl_trx (claim-level TRx for decile computation) ─────────────────────────
pt_lvl_trx = spark.table(f"`{CATALOG}`.`{SCHEMA}`.pt_lvl_trx")

trx_window = pt_lvl_trx.filter(
    (F.col("SVC_DT") >= LOOKBACK_START) &
    (F.col("SVC_DT") <= LOOKBACK_END)
)

# ── NPI mapping for Targeting/BPT ──────────────────────────────────────────────
npi_provider_bhid_mapping = spark.table(
    f"`{CATALOG}`.kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING"
)

print(f"lrx_window rows : {lrx_window.count():,}")
print(f"trx_window rows : {trx_window.count():,}")

---
### Criterion 1 — Per-Product-Class Decile 2+ (TRx-Based)

For each of the 5 target product classes, independently:
1. Count **TRx** (claims) per `PROVIDER_ID` within that class from `pt_lvl_trx`
2. Rank into deciles via cumulative TRx (each decile holds ~10% of total TRx volume) — **matching the v1/NBA 3.0 methodology**
3. Keep Decile 2+

Then combine (intersection or union) the 5 Decile 2+ sets.

In [0]:
# ── Compute TRx-based Decile 2+ HCPs per product class ───────────────────────
# Matches v1/NBA 3.0 methodology:
#   - Metric: TRx (each claim = 1 TRx)
#   - Method: Cumulative TRx-based (each decile holds ~10% of total TRx volume)
#   - Source: pt_lvl_trx (claim-level)

def get_decile2plus_hcps_trx(trx_df, product_class, cutoff=DECILE_CUTOFF):
    """Compute TRx-based cumulative deciles for a PRODUCT_CLASS, return (all_deciles, d2plus)."""
    # Step 1: Count claims (TRx) per PROVIDER_ID for this product class
    pc_df = trx_df.filter(F.col("PRODUCT_CLASS") == product_class)
    hcp_trx = pc_df.groupBy("PROVIDER_ID").agg(F.count("*").alias("MARKET_TRX"))

    # Step 2: Cumulative TRx-based decile (matching v1 methodology)
    # Sort by MARKET_TRX ascending, compute cumulative sum, assign decile
    w = Window.orderBy(F.asc("MARKET_TRX"))
    total_trx = hcp_trx.agg(F.sum("MARKET_TRX")).collect()[0][0]
    trx_per_decile = total_trx / 10

    hcp_trx = hcp_trx.withColumn("CUMULATIVE_TRX", F.sum("MARKET_TRX").over(w))
    hcp_trx = hcp_trx.withColumn(
        "DECILE",
        F.least(
            (F.col("CUMULATIVE_TRX") / F.lit(trx_per_decile)).cast("int") + 1,
            F.lit(10)
        )
    )
    hcp_trx = hcp_trx.drop("CUMULATIVE_TRX")

    d2plus = hcp_trx.filter(F.col("DECILE") >= cutoff)
    return hcp_trx, d2plus

# Store per-class results
class_results = {}
for pc in TARGET_PRODUCT_CLASSES:
    all_deciles, d2plus = get_decile2plus_hcps_trx(trx_window, pc)
    total_hcps = all_deciles.count()
    d2_count = d2plus.count()
    class_results[pc] = {
        "all_deciles": all_deciles,
        "d2plus": d2plus,
        "total": total_hcps,
        "d2plus_count": d2_count,
    }
    print(f"{pc:<8}  Total HCPs: {total_hcps:>10,}  |  Decile 2+: {d2_count:>10,}  |  Dropped (D1): {total_hcps - d2_count:>10,}")

In [0]:
# ── Intersection: HCPs Decile 2+ in ALL 5 product classes ──────────────────────
pc_list = list(TARGET_PRODUCT_CLASSES)
intersect_df = class_results[pc_list[0]]["d2plus"].select("PROVIDER_ID").distinct()
for pc in pc_list[1:]:
    intersect_df = intersect_df.join(
        class_results[pc]["d2plus"].select("PROVIDER_ID").distinct(),
        on="PROVIDER_ID", how="inner"
    )

# ── Union: HCPs Decile 2+ in ANY of the 5 product classes ─────────────────────
union_df = class_results[pc_list[0]]["d2plus"].select("PROVIDER_ID")
for pc in pc_list[1:]:
    union_df = union_df.unionByName(class_results[pc]["d2plus"].select("PROVIDER_ID"))
union_df = union_df.distinct()

intersect_count = intersect_df.count()
union_count = union_df.count()
print(f"Intersection (all 5 Decile 2+): {intersect_count:,}")
print(f"Union (any 1 Decile 2+)       : {union_count:,}")
print(f"Intersection / Union ratio    : {100*intersect_count/union_count:.1f}%")

# ══════════════════════════════════════════════════════════════════════════════
# ▶ SWITCH: Use UNION as the primary Criterion 1 set for v3
# ══════════════════════════════════════════════════════════════════════════════
criterion1_hcps = union_df
print(f"\n✔ Using UNION ({union_count:,} HCPs) as Criterion 1")

In [0]:
# ── Pairwise overlap counts between product classes ─────────────────────────────
import itertools
import builtins

print(f"{'Pair':<20} {'Overlap':>10} {'% of smaller set':>18}")
print("-" * 50)
for a, b in itertools.combinations(TARGET_PRODUCT_CLASSES, 2):
    ids_a = class_results[a]["d2plus"].select("PROVIDER_ID").distinct()
    ids_b = class_results[b]["d2plus"].select("PROVIDER_ID").distinct()
    overlap = ids_a.join(ids_b, on="PROVIDER_ID", how="inner").count()
    smaller = builtins.min(class_results[a]["d2plus_count"], class_results[b]["d2plus_count"])
    pct = 100 * overlap / smaller if smaller else 0
    print(f"{a+' ∩ '+b:<20} {overlap:>10,} {pct:>17.1f}%")

In [0]:
# ── Enrich Criterion 1 (union) HCPs with per-class deciles ────────────────────
criterion1_enriched = criterion1_hcps
for pc in TARGET_PRODUCT_CLASSES:
    col_name = f"{pc.upper().replace(' ','')}_DECILE"
    decile_df = class_results[pc]["all_deciles"].select(
        "PROVIDER_ID", F.col("DECILE").alias(col_name)
    )
    criterion1_enriched = criterion1_enriched.join(decile_df, on="PROVIDER_ID", how="left")

# Add flags: which classes is this HCP Decile 2+ in?
criterion1_enriched = (
    criterion1_enriched
    .withColumn("PRODUCT_CLASS_UNION_FLAG", F.lit(1))
    .withColumn("IN_ALL_5", F.when(
        (F.col("SGLT2_DECILE") >= DECILE_CUTOFF) &
        (F.col("GLP1_DECILE") >= DECILE_CUTOFF) &
        (F.col("ACE_DECILE") >= DECILE_CUTOFF) &
        (F.col("ARB_DECILE") >= DECILE_CUTOFF) &
        (F.col("SMRA_DECILE") >= DECILE_CUTOFF),
        1
    ).otherwise(0))
    .withColumn("N_CLASSES_D2PLUS",
        (F.when(F.col("SGLT2_DECILE") >= DECILE_CUTOFF, 1).otherwise(0)) +
        (F.when(F.col("GLP1_DECILE") >= DECILE_CUTOFF, 1).otherwise(0)) +
        (F.when(F.col("ACE_DECILE") >= DECILE_CUTOFF, 1).otherwise(0)) +
        (F.when(F.col("ARB_DECILE") >= DECILE_CUTOFF, 1).otherwise(0)) +
        (F.when(F.col("SMRA_DECILE") >= DECILE_CUTOFF, 1).otherwise(0))
    )
    .fillna(0)
)

print(f"Criterion 1 (union) HCPs: {criterion1_enriched.count():,}")
print(f"\nHCPs by # of product classes at Decile 2+:")
criterion1_enriched.groupBy("N_CLASSES_D2PLUS").agg(
    F.count("*").alias("n_hcps")
).orderBy("N_CLASSES_D2PLUS").show()

In [0]:
# ── Kerendia writers (full history, same as v2) ────────────────────────────────
kerendia_hcps = (
    lrx_nbrx
    .filter(
        (F.col("PRODUCT_CLASS") == "KERENDIA") &
        (F.col("WEEK_ENDING_DATE") <= LOOKBACK_END)
    )
    .select("PROVIDER_ID").distinct()
    .withColumn("KERENDIA_FLAG", F.lit(1))
)
print(f"Kerendia Writers: {kerendia_hcps.count():,}")

In [0]:
# ── Specialty HCPs (same as v2) ────────────────────────────────────────────────
specialty_hcps = lrx_window.filter(F.col("SPECIALTY_GROUP").isin(TARGET_SPECIALTIES)).groupBy("PROVIDER_ID").agg(
    F.max(F.when(F.col("SPECIALTY_GROUP") == "NEPH",        1).otherwise(0)).alias("NEPH_FLAG"),
    F.max(F.when(F.col("SPECIALTY_GROUP") == "CARD",        1).otherwise(0)).alias("CARD_FLAG"),
    F.max(F.when(F.col("SPECIALTY_GROUP") == "ENDO",        1).otherwise(0)).alias("ENDO_FLAG"),
    F.max(F.when(F.col("SPECIALTY_GROUP") == "PCP",         1).otherwise(0)).alias("PCP_FLAG"),
    F.max(F.when(F.col("SPECIALTY_GROUP") == "HOSPITALIST", 1).otherwise(0)).alias("HOSPITALIST_FLAG"),
).withColumn("SPECIALTY_HCP_FLAG", F.lit(1))
print(f"Specialty HCPs: {specialty_hcps.count():,}")

In [0]:
# ── Targeting HCPs (same as v1/v2) ───────────────────────────────────────────
targeting_hcps = (
    Get_Data_Snowflakes("""
        SELECT DISTINCT BAYER_ID AS BH_ID
        FROM PHCDW.PHCDW_TARGETING.CP_KER_HCP
        WHERE FF IN ('CR1', 'CR2', 'CR4', 'CR5')
          AND (UPPER(PRODUCT_1) = 'KERENDIA' OR UPPER(PRODUCT_2) = 'KERENDIA')
    """)
    .join(npi_provider_bhid_mapping.select("PROVIDER_ID", "BH_ID").distinct(), on="BH_ID", how="inner")
    .select("PROVIDER_ID").distinct()
    .withColumn("TARGETING_HCP_FLAG", F.lit(1))
)

# ── BPT HCPs (same source, separate flag) ───────────────────────────────────
bpt_hcps = (
    Get_Data_Snowflakes("""
        SELECT DISTINCT BAYER_ID AS BH_ID
        FROM PHCDW.PHCDW_TARGETING.CP_KER_HCP
        WHERE FF IN ('CR1', 'CR2', 'CR4', 'CR5')
          AND (UPPER(PRODUCT_1) = 'KERENDIA' OR UPPER(PRODUCT_2) = 'KERENDIA')
    """)
    .join(npi_provider_bhid_mapping.select("PROVIDER_ID", "BH_ID").distinct(), on="BH_ID", how="inner")
    .select("PROVIDER_ID").distinct()
    .withColumn("BPT_HCP_FLAG", F.lit(1))
)
print(f"Targeting HCPs: {targeting_hcps.count():,}")
print(f"BPT HCPs      : {bpt_hcps.count():,}")

In [0]:
# ── Full outer join all criteria ─────────────────────────────────────────────────────
def outer_join(df_a: DataFrame, df_b: DataFrame) -> DataFrame:
    return df_a.join(df_b, on="PROVIDER_ID", how="outer")

dfs = [
    criterion1_enriched, # per-class deciles + PRODUCT_CLASS_UNION_FLAG + IN_ALL_5 + N_CLASSES_D2PLUS
    kerendia_hcps,       # KERENDIA_FLAG
    specialty_hcps,      # SPECIALTY_HCP_FLAG + per-specialty flags
    targeting_hcps,      # TARGETING_HCP_FLAG
    bpt_hcps,            # BPT_HCP_FLAG
]

df_hcp_universe = reduce(outer_join, dfs)
df_hcp_universe = df_hcp_universe.fillna(0)
df_hcp_universe = df_hcp_universe.dropDuplicates(["PROVIDER_ID"])

print(f"Total HCP Universe v3: {df_hcp_universe.count():,}")
print(f"Columns: {df_hcp_universe.columns}")

In [0]:
# ── Flag totals ──────────────────────────────────────────────────────────────────────
flag_cols = [c for c in df_hcp_universe.columns if c.endswith("_FLAG")]
agg_exprs = [F.sum(c).alias(c) for c in flag_cols]
flag_totals = df_hcp_universe.agg(*agg_exprs).collect()[0].asDict()

total = df_hcp_universe.count()
print(f"{'Criterion / Flag':<40} {'Count':>10} {'% of Universe':>15}")
print("-" * 68)
for col_name, cnt in sorted(flag_totals.items(), key=lambda x: -int(x[1])):
    pct = 100 * int(cnt) / total
    label = col_name.replace("_FLAG", "").replace("_", " ")
    print(f"{label:<40} {int(cnt):>10,} {pct:>14.1f}%")
print("-" * 68)
print(f"{'TOTAL HCP UNIVERSE v3':<40} {total:>10,}")

In [0]:
# ── Compare v1, v2, v3 ─────────────────────────────────────────────────────────
v1_path = (
    f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_volume/"
    "hcp_level_model/dev_folder_3_1/notebook_output/"
    "kerendia_nba_3_hcp_model_base_hcp_universe_for_inference"
)
v2_path = (
    f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_volume/"
    "hcp_level_model/dev_folder_3_1/notebook_output/"
    "kerendia_nba_3_hcp_universe_v2"
)

v1_ids = spark.read.format("delta").load(v1_path).select("PROVIDER_ID").distinct()
v2_ids = spark.read.format("delta").load(v2_path).select("PROVIDER_ID").distinct()
v3_ids = df_hcp_universe.select("PROVIDER_ID").distinct()

v1_n, v2_n, v3_n = v1_ids.count(), v2_ids.count(), v3_ids.count()

v3_v1_overlap = v3_ids.join(v1_ids, on="PROVIDER_ID", how="inner").count()
v3_v2_overlap = v3_ids.join(v2_ids, on="PROVIDER_ID", how="inner").count()

print(f"{'Version':<12} {'Universe':>12} {'v3 overlap':>12} {'% of v3':>10}")
print("-" * 48)
print(f"{'v1 (product)':<12} {v1_n:>12,} {v3_v1_overlap:>12,} {100*v3_v1_overlap/v3_n:>9.1f}%")
print(f"{'v2 (indication)':<12} {v2_n:>12,} {v3_v2_overlap:>12,} {100*v3_v2_overlap/v3_n:>9.1f}%")
print(f"{'v3 (prod-class)':<12} {v3_n:>12,}")
print(f"\nv3 ∩ v1 as % of v1: {100*v3_v1_overlap/v1_n:.1f}%")
print(f"v3 ∩ v2 as % of v2: {100*v3_v2_overlap/v2_n:.1f}%")

In [0]:
curr_date = dt.today().strftime("%Y_%m_%d")
df_hcp_universe = df_hcp_universe.withColumn("PULL_DT", F.lit(curr_date))

(
    df_hcp_universe
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(OUTPUT_PATH)
)
print(f"Saved to: {OUTPUT_PATH}")
print(f"Final HCP Universe v3 size: {df_hcp_universe.count():,}")